[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TianshuangQiu/TorchCode/blob/master/solutions/45_per_group_linear_solution.ipynb)

# 🔴 Solution: Per-Group Linear (MoE Forward)

**Primitive: fancy indexing + `bmm` (batched matrix multiply)**

**Reduction:** `y[i] = W[group_ids[i]] @ x[i] + b[group_ids[i]]`.

Key steps:
1. `W[group_ids]` — fancy indexing gathers the right `(d_out, d_in)` matrix for each token → shape `(N, d_out, d_in)`
2. `x.unsqueeze(-1)` → `(N, d_in, 1)` — column vectors
3. `torch.bmm(W_per, x_col).squeeze(-1)` → `(N, d_out)` — batched matrix-vector multiply
4. `+ b[group_ids]` — fancy indexing again for the bias

An equivalent formulation: `torch.einsum('noi,ni->no', W[group_ids], x) + b[group_ids]`

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def per_group_linear(
    x: torch.Tensor,
    group_ids: torch.Tensor,
    W: torch.Tensor,
    b: torch.Tensor,
) -> torch.Tensor:
    # primitive: fancy indexing gathers per-token weights; bmm does batched mat-vec multiply
    W_per = W[group_ids]                                    # (N, d_out, d_in)
    out = torch.bmm(W_per, x.unsqueeze(-1)).squeeze(-1)     # (N, d_out)
    return out + b[group_ids]                               # (N, d_out)

In [ ]:
# Verify
x         = torch.tensor([[1.,0.],[0.,1.],[1.,1.]])
group_ids = torch.tensor([0, 1, 0])
W = torch.zeros(2, 2, 2)
W[0] = torch.eye(2)
W[1] = 2 * torch.eye(2)
b = torch.zeros(2, 2)
b[1] = torch.tensor([1., 1.])

result = per_group_linear(x, group_ids, W, b)
print('output:', result.tolist())
print('expect: [[1.0, 0.0], [1.0, 3.0], [1.0, 1.0]]')

# Gradient check
W2 = W.clone().requires_grad_(True)
b2 = b.clone().requires_grad_(True)
per_group_linear(x, group_ids, W2, b2).sum().backward()
print('W.grad:', W2.grad)
print('b.grad:', b2.grad)

In [ ]:
import torch, time

# ── Test 1: spec example (N=3, G=2, d_in=2, d_out=2) ─────────────────────
x         = torch.tensor([[1.,0.],[0.,1.],[1.,1.]])
group_ids = torch.tensor([0, 1, 0])
W = torch.zeros(2, 2, 2)
W[0] = torch.eye(2)
W[1] = 2 * torch.eye(2)
b = torch.zeros(2, 2)
b[1] = torch.tensor([1., 1.])
result = per_group_linear(x, group_ids, W, b)
expected = torch.tensor([[1.,0.],[1.,3.],[1.,1.]])
assert result.shape == (3, 2), f"Shape: {result.shape}"
assert torch.allclose(result, expected), f"Got {result}, expected {expected}"
print("Test 1 passed: spec example")

# ── Test 2: single expert (G=1) equals plain linear ───────────────────────
torch.manual_seed(42)
N, d_in, d_out = 8, 4, 6
x = torch.randn(N, d_in)
W = torch.randn(1, d_out, d_in)
b = torch.randn(1, d_out)
group_ids = torch.zeros(N, dtype=torch.long)
result = per_group_linear(x, group_ids, W, b)
expected = (x @ W[0].T) + b[0]
assert result.shape == (N, d_out), f"Shape: {result.shape}"
assert torch.allclose(result, expected, atol=1e-5), "Mismatch vs plain linear"
print("Test 2 passed: single expert equals linear")

# ── Test 3: one token per group (N=G) ─────────────────────────────────────
torch.manual_seed(7)
G, d_in, d_out = 5, 3, 4
x = torch.randn(G, d_in)
W = torch.randn(G, d_out, d_in)
b = torch.randn(G, d_out)
group_ids = torch.arange(G)
result = per_group_linear(x, group_ids, W, b)
assert result.shape == (G, d_out), f"Shape: {result.shape}"
for i in range(G):
    expected_i = W[i] @ x[i] + b[i]
    assert torch.allclose(result[i], expected_i, atol=1e-5), f"Token {i} mismatch"
print("Test 3 passed: one token per group")

# ── Test 4: gradients flow to W and b ────────────────────────────────────
torch.manual_seed(0)
N, G, d_in, d_out = 4, 2, 3, 3
x = torch.randn(N, d_in)
W = torch.randn(G, d_out, d_in, requires_grad=True)
b = torch.randn(G, d_out, requires_grad=True)
group_ids = torch.tensor([0, 1, 0, 1])
out = per_group_linear(x, group_ids, W, b)
out.sum().backward()
assert W.grad is not None, "W.grad is None"
assert b.grad is not None, "b.grad is None"
assert W.grad.shape == W.shape, f"W.grad shape: {W.grad.shape}"
assert b.grad.shape == b.shape, f"b.grad shape: {b.grad.shape}"
print("Test 4 passed: gradients flow")

# ── Test 5: large N=1000 G=8 d_in=128 d_out=256 (timing) ─────────────────
torch.manual_seed(0)
N, G, d_in, d_out = 1000, 8, 128, 256
x = torch.randn(N, d_in)
W = torch.randn(G, d_out, d_in)
b = torch.randn(G, d_out)
group_ids = torch.randint(0, G, (N,))
t0 = time.time()
result = per_group_linear(x, group_ids, W, b)
elapsed = time.time() - t0
assert result.shape == (N, d_out), f"Shape: {result.shape}"
assert elapsed < 2.0, f"Too slow: {elapsed:.2f}s (expected <2s — no loops)"
print(f"Test 5 passed: large input timing ({elapsed:.3f}s)")

print("\nAll tests passed!")
